In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [2]:
np.random.seed(0)

In [3]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [4]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [5]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

In [6]:
def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [7]:
def accuracy_loss(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    losses = np.empty_like(thresholds)
    for i, threshold in enumerate(thresholds):
        Y_p = (X_p >= threshold).astype(float)
        losses[i] = np.abs(Y_true - Y_p).mean()
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [8]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    acc_loss_p = accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p
    return acc_loss_p

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [9]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [11]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            X, partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [10]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)

def find_partitions_greedy(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1


    Q = collections.deque(itertools.combinations(P.keys(), 2))

    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for c_id in P.keys():
                if c_id != new_id:
                    Q.append((new_id, c_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
    return list(P.values())

In [ ]:
def approximation_ratio(loss_optimal, loss_greedy):
    if (1 - loss_greedy) == 0:
        return np.nan
    return (1 - loss_optimal) / (1 - loss_greedy)

In [12]:
threshold_true = 0.9999999
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
X = np.arange(threshold_min, threshold_max+1e-4, 1e-4).round(4)

In [13]:
runs = 100
N = np.arange(2, 11)
# C = np.arange(0., 10, 0.5) + 0.5
C = [0.5]

results = {"n": [], "run": [], "thresholds": [], "priors": [], "threshold_true": [], "c": [], "opt": [], "greedy": [], "acc_loss_opt": [], "acc_loss_greedy": [], "ratio": []}

for n in N:
    for run in tqdm.trange(runs, desc=f"[ n={n} ]"):
        thresholds = np.sort(np.random.rand(n)).round(6)
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        diff = 1 - np.sum(priors.round(6))
        priors = priors.round(6)
        priors[0] += diff

        for c in C:
            partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
            partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)

            acc_loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
            acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)

            ratio = (1 - acc_loss_opt)/(1- acc_loss_greedy)

            results["n"].append(n)
            results["run"].append(run)
            results["thresholds"].append(deepcopy(thresholds))
            results["priors"].append(deepcopy(priors))
            results["threshold_true"].append(threshold_true)
            results["c"].append(c)
            results["opt"].append(deepcopy(partition_opt))
            results["greedy"].append(deepcopy(partition_greedy))
            results["acc_loss_opt"].append(acc_loss_opt.item())
            results["acc_loss_greedy"].append(acc_loss_greedy.item())
            results["ratio"].append(ratio.item())

[ n=10 ]: 100%|██████████| 100/100 [01:53<00:00,  1.14s/it]


In [14]:
df = pd.DataFrame(results)
print(df.shape)
df.head()

(900, 11)


,n,run,thresholds,priors,threshold_true,c,opt,greedy,acc_loss_opt,acc_loss_greedy,ratio
0,2,0,"[0.548814, 0.715189]","[0.525217, 0.474783]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.9999,0.9999,1.0
1,2,1,"[0.423655, 0.645894]","[0.329171, 0.670829]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.9999,0.9999,1.0
2,2,2,"[0.383442, 0.963663]","[0.59951, 0.40049]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.9999,0.9999,1.0
3,2,3,"[0.568045, 0.925597]","[0.449125, 0.550875]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.9999,0.9999,1.0
4,2,4,"[0.020218, 0.83262]","[0.472134, 0.527866]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.9999,0.9999,1.0


In [15]:
df_im = df.copy()
df_im["n"] = df_im["n"].astype(str)
df_im["c"] = df_im["c"].astype(str)
px.scatter(df_im, x="n", y="ratio", color="c", hover_data=["run"])

In [16]:
px.scatter(df, x="acc_loss_opt", y="ratio", color="acc_loss_greedy")#.update_traces(marker=dict(size=3))

In [17]:
i_max = df["ratio"].argmax()
df_max = df.iloc[[i_max]]
df_max

,n,run,thresholds,priors,threshold_true,c,opt,greedy,acc_loss_opt,acc_loss_greedy,ratio
662,8,62,"[0.067983, 0.085424, 0.125552, 0.158896, 0.399...","[0.06613600000000003, 0.188222, 0.169938, 0.10...",1.0,0.5,"[[6], [0, 1, 2, 3, 4, 5, 7]]","[[0, 1, 2, 3, 4, 5, 6, 7]]",0.875634,0.9999,1243.787361


In [18]:
thresholds = df_max["thresholds"].item()
priors = df_max["priors"].item()

threshold_true = df_max["threshold_true"].item()
c = df_max["c"].item()

# partition_opt = df_max["opt"].item()
# partition_greedy = df_max["greedy"].item()

# acc_loss_opt = df_max["acc_loss_opt"].item()
# acc_loss_greedy = df_max["acc_loss_greedy"].item()

partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)

acc_loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)

In [107]:
print(f"c         : {c:.4f}")
print(f"threshold true: {threshold_true}")
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)

print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.7f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_opt)}")
print(f"Acc Loss : {acc_loss_opt:.7f}")
print()
print(f"Ratio: {(1 - acc_loss_opt) / (1 - acc_loss_greedy):.4f}")

c         : 0.5000
threshold true: 0.9999999


,0,1,2,3,4,5,6,7
Thresholds,0.067983,0.085424,0.125552,0.158896,0.399359,0.431586,0.587459,0.982314
Priors,0.066136,0.188222,0.169938,0.102170,0.013735,0.100752,0.123847,0.235200


Greedy
------
Partition: [[0, 1, 2, 3, 4, 5, 6, 7]]
Acc Loss : 0.9999000

Optimal
-------
Partition: [[0, 1, 2, 3, 4, 5, 7], [6]]
Acc Loss : 0.8756337

Ratio: 1243.7874


In [134]:
a, b = [0,1,2,3,4,5,7], [6]

acc_loss_a, thresholds_a, priors_a = evaluate_partition(X, a, thresholds, priors, threshold_true, 0.59, True)
acc_loss_b, thresholds_b, priors_b = evaluate_partition(X, b, thresholds, priors, threshold_true, 0.59, True)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, 0.59, True)
rhs = acc_loss_ab * np.sum(priors[ab])

print(f"    threshold true: {threshold_true}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"            merge?: {lhs - rhs > -1e-9}")
print()

print(f"          priors a: {priors_a.round(4)}")
print(f"      thresholds a: {thresholds_a.round(4)}")
print(f"          priors b: {priors_b.round(4)}")
print(f"      thresholds b: {thresholds_b.round(4)}")
print(f"     thresholds ab: {thresholds_ab.round(4)}")
print(f"         priors ab: {priors_ab.round(4)}")

    threshold true: 0.9999999
                 c: 0.5
                 a: [0, 1, 2, 3, 4, 5, 7]
                 b: [6]
   accuracy loss a: 0.8308715
   accuracy loss b: 0.9999000
  accuracy loss ab: 0.9147671
               LHS: 0.8518052
               RHS: 0.9147671
            merge?: False

          priors a: [0.5265 0.1145 0.2352]
      thresholds a: [0.1589 0.4316 0.9823]
          priors b: [0.1238]
      thresholds b: [0.5875]
     thresholds ab: [0.1589 0.9823]
         priors ab: [0.5265 0.4735]


In [63]:
C = np.arange(0.01, 1.0+0.01, 0.01).round(2)
thresholds_true = np.arange(0.1, 1.0, 0.1).round(1)
results = {"c": [], "threshold_true": [], "partition_opt": [], "partition_greedy": [], "acc_loss_opt": [], "acc_loss_greedy": [], "ratio": []}

for threshold_true in thresholds_true:
    for c in tqdm.tqdm(C, desc=f" [t*={threshold_true}]"):
        partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)

        acc_loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
        acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)

        ratio = (1 - acc_loss_opt) / (1 - acc_loss_greedy)

        results["c"].append(c.item())
        results["threshold_true"].append(threshold_true.item())
        results["partition_opt"].append(partition_opt)
        results["partition_greedy"].append(partition_greedy)
        results["acc_loss_opt"].append(acc_loss_opt.item())
        results["acc_loss_greedy"].append(acc_loss_greedy.item())
        results["ratio"].append(ratio.item())

 [t*=0.9]: 100%|██████████| 100/100 [00:20<00:00,  4.84it/s]


In [40]:
px.line(results, x="c", y="ratio")